In [1]:
# ============================================================
# AI Compliance Audit Agent – Capstone Project (Python)
# ============================================================
# Features:
# - Rule-based tool for policy / privacy violation detection
# - Risk scoring engine (0–100)
# - LLM-powered "AI Brain" (Cohere Chat API or mock)
# - Human-readable report generator
# - Simple session memory (in-memory store)
# - Logging + metrics (observability)
# - Simple evaluation runner
# - Gradio UI for interactive use
# ============================================================

!pip install -q cohere gradio

import os
import re
import json
import time
import logging
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import gradio as gr
import cohere

# ------------------------------
# CONFIG & LOGGING
# ------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s",
)

COHERE_API_KEY = os.environ.get("COHERE_API_KEY", None)
USE_MOCK_LLM = COHERE_API_KEY is None  # Set to False if you definitely want live LLM

MODEL_NAME = "command-a-03-2025"  # Adjust if needed per Cohere docs

if not USE_MOCK_LLM:
    co_client = cohere.ClientV2(COHERE_API_KEY)
    logging.info("Using Cohere Chat API.")
else:
    co_client = None
    logging.warning("COHERE_API_KEY not set – using MOCK LLM responses for testing only.")


# ============================================================
# DATA STRUCTURES
# ============================================================

@dataclass
class Violation:
    category: str
    description: str
    span: str
    start_idx: int
    end_idx: int


@dataclass
class ComplianceAnalysisResult:
    text: str
    violations: List[Violation]
    risk_score: int
    risk_level: str
    safe_rewrite: str
    recommendations: List[str]
    narrative_summary: str
    timestamp: float


# ============================================================
# RULE CHECKER TOOL
# ============================================================

class RuleChecker:
    """
    Simple heuristic / regex-based rule checker.
    This is a 'custom tool' that the agent uses before calling the LLM.
    """

    def __init__(self):
        # Regexes and keyword lists for demo purposes
        self.patterns = {
            "phone_number": re.compile(r"\b(\+?\d[\d\-\s]{7,}\d)\b"),
            "email": re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"),
            "ip_address": re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
        }
        self.keyword_categories = {
            "financial_data": [
                "credit card", "debit card", "cvv", "cvc",
                "bank account", "account number", "routing number",
            ],
            "health_data": [
                "medical record", "diagnosis", "prescription",
                "mental health", "HIV status", "blood test",
            ],
            "secrets": [
                "password", "secret key", "api key", "access token",
                "private key", "ssh key",
            ],
            "confidential": [
                "confidential", "internal only", "do not distribute",
                "proprietary", "trade secret",
            ],
        }

    def analyze(self, text: str) -> List[Violation]:
        violations: List[Violation] = []

        # Pattern-based detection
        for name, pattern in self.patterns.items():
            for match in pattern.finditer(text):
                violations.append(
                    Violation(
                        category=name,
                        description=f"Detected pattern: {name.replace('_', ' ')}",
                        span=match.group(0),
                        start_idx=match.start(),
                        end_idx=match.end(),
                    )
                )

        # Keyword-based detection
        lowered = text.lower()
        for category, keywords in self.keyword_categories.items():
            for kw in keywords:
                start = 0
                while True:
                    idx = lowered.find(kw, start)
                    if idx == -1:
                        break
                    span = text[idx:idx+len(kw)]
                    violations.append(
                        Violation(
                            category=category,
                            description=f"Keyword '{kw}' suggests {category.replace('_', ' ')}",
                            span=span,
                            start_idx=idx,
                            end_idx=idx + len(kw),
                        )
                    )
                    start = idx + len(kw)

        return violations


# ============================================================
# RISK SCORING ENGINE
# ============================================================

class RiskScorer:
    """
    Converts detected violations into a 0–100 risk score.
    """

    category_weights = {
        "phone_number": 10,
        "email": 8,
        "ip_address": 6,
        "financial_data": 25,
        "health_data": 25,
        "secrets": 30,
        "confidential": 15,
    }

    @staticmethod
    def score(violations: List[Violation]) -> int:
        score = 0
        for v in violations:
            w = RiskScorer.category_weights.get(v.category, 5)
            score += w

        score = min(100, score)  # cap at 100
        return int(score)

    @staticmethod
    def risk_level(score: int) -> str:
        if score >= 70:
            return "HIGH"
        elif score >= 40:
            return "MEDIUM"
        elif score > 0:
            return "LOW"
        return "NONE"


# ============================================================
# SIMPLE IN-MEMORY SESSION & METRICS
# ============================================================

class SessionMemory:
    """
    Simple in-memory session manager for demo.
    - Stores last N analyses per session_id
    - Keeps basic aggregate metrics
    """

    def __init__(self, max_history: int = 10):
        self.sessions: Dict[str, List[ComplianceAnalysisResult]] = {}
        self.max_history = max_history
        self.metrics = {
            "total_requests": 0,
            "avg_risk_score": 0.0,
        }

    def add_result(self, session_id: str, result: ComplianceAnalysisResult):
        if session_id not in self.sessions:
            self.sessions[session_id] = []
        self.sessions[session_id].append(result)
        if len(self.sessions[session_id]) > self.max_history:
            self.sessions[session_id] = self.sessions[session_id][-self.max_history :]

        # Update metrics
        self.metrics["total_requests"] += 1
        # Recompute average risk (simple running average)
        n = self.metrics["total_requests"]
        prev_avg = self.metrics["avg_risk_score"]
        new_avg = prev_avg + (result.risk_score - prev_avg) / n
        self.metrics["avg_risk_score"] = new_avg

    def get_history(self, session_id: str) -> List[ComplianceAnalysisResult]:
        return self.sessions.get(session_id, [])

    def get_metrics(self) -> Dict[str, Any]:
        return self.metrics


session_memory = SessionMemory()


# ============================================================
# LLM "BRAIN" – COHERE OR MOCK
# ============================================================

def call_llm_brain(
    text: str,
    violations: List[Violation],
    risk_score: int,
    risk_level: str,
) -> Dict[str, Any]:
    """
    Calls Cohere Chat API (or mock) to:
    - rewrite text safely
    - explain risks
    - give recommendations
    - summarize as narrative
    Returns dict with keys: safe_rewrite, recommendations, narrative_summary
    """

    violations_json = [asdict(v) for v in violations]

    system_prompt = (
        "You are a strict AI Compliance and Safety Auditor. "
        "Given user content, detected violations, and a risk score, you must:\n"
        "1) Rewrite the text into a safe, compliant version.\n"
        "2) Provide 3–6 concrete recommendations to reduce risk.\n"
        "3) Write a concise narrative summary explaining why this is risky.\n"
        "Return JSON ONLY with keys: safe_rewrite, recommendations, narrative_summary."
    )

    user_payload = {
        "input_text": text,
        "violations": violations_json,
        "risk_score": risk_score,
        "risk_level": risk_level,
    }

    if USE_MOCK_LLM:
        # MOCK: deterministic simple behavior, no external calls
        logging.info("Using MOCK LLM brain.")
        safe_text = (
            "[SAFE VERSION – MOCK]\n"
            "This text has been redacted to remove obvious sensitive data such as "
            "phone numbers, emails, or explicit secrets."
        )
        recs = [
            "Remove or mask personal identifiers (e.g., phone numbers, emails).",
            "Avoid including financial data or authentication secrets in plain text.",
            "Add an internal review step before publishing sensitive documents.",
        ]
        summary = (
            "The original text appears to include sensitive or confidential information. "
            "To stay compliant, remove personal data, secrets, and internal-only statements "
            "before sharing externally."
        )
        return {
            "safe_rewrite": safe_text,
            "recommendations": recs,
            "narrative_summary": summary,
        }

    # REAL LLM CALL (Cohere Chat API)
    logging.info("Calling Cohere Chat API for compliance rewrite & explanation.")

    # Cohere v2 Chat API example per docs
    response = co_client.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": json.dumps(user_payload),
            },
        ],
        temperature=0.2,
    )

    # Cohere returns a rich object; extract text
    # For v2, main content is in response.message.content
    if hasattr(response, "message"):
        raw_text = "".join(
            [block.text for block in response.message.content if getattr(block, "type", "text") == "text"]
        )
    else:
        # Fallback for older SDK versions
        raw_text = str(response)

    # Try to parse JSON from the model output
    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError:
        logging.warning("LLM did not return valid JSON; wrapping as narrative only.")
        parsed = {
            "safe_rewrite": "[LLM ERROR] Could not parse JSON. Please manually review.",
            "recommendations": [
                "Manually review the content for sensitive data.",
                "Ensure personal identifiers and secrets are removed.",
            ],
            "narrative_summary": raw_text,
        }

    # Normalize fields
    safe_rewrite = parsed.get("safe_rewrite", "[missing safe_rewrite]")
    recommendations = parsed.get("recommendations", [])
    if isinstance(recommendations, str):
        recommendations = [recommendations]
    narrative_summary = parsed.get("narrative_summary", "")

    return {
        "safe_rewrite": safe_rewrite,
        "recommendations": recommendations,
        "narrative_summary": narrative_summary,
    }


# ============================================================
# MAIN AGENT ORCHESTRATOR
# ============================================================

class AIComplianceAuditAgent:
    """
    High-level orchestrator:
    - Uses rule-based tool for violation detection
    - Uses risk scoring engine
    - Calls LLM "brain"
    - Returns structured result
    """

    def __init__(self, rule_checker: RuleChecker, risk_scorer: RiskScorer):
        self.rule_checker = rule_checker
        self.risk_scorer = risk_scorer

    def analyze_text(self, text: str, session_id: str = "default") -> ComplianceAnalysisResult:
        start_time = time.time()
        logging.info("Starting compliance analysis.")

        # 1) Rule checker (tool)
        violations = self.rule_checker.analyze(text)
        logging.info(f"Detected {len(violations)} potential violations.")

        # 2) Risk score
        risk_score = self.risk_scorer.score(violations)
        risk_level = self.risk_scorer.risk_level(risk_score)
        logging.info(f"Risk score: {risk_score} ({risk_level})")

        # 3) LLM brain for safe rewrite, recommendations, narrative
        llm_output = call_llm_brain(text, violations, risk_score, risk_level)

        result = ComplianceAnalysisResult(
            text=text,
            violations=violations,
            risk_score=risk_score,
            risk_level=risk_level,
            safe_rewrite=llm_output["safe_rewrite"],
            recommendations=llm_output["recommendations"],
            narrative_summary=llm_output["narrative_summary"],
            timestamp=start_time,
        )

        # 4) Save to session memory
        session_memory.add_result(session_id, result)

        logging.info("Analysis complete.")
        return result


agent = AIComplianceAuditAgent(RuleChecker(), RiskScorer())


# ============================================================
# HUMAN-READABLE REPORT FORMATTER
# ============================================================

def format_report(result: ComplianceAnalysisResult) -> str:
    lines = []
    lines.append("=== AI COMPLIANCE AUDIT REPORT ===")
    lines.append(f"Risk Score: {result.risk_score} / 100  |  Level: {result.risk_level}")
    lines.append("")
    lines.append("Detected Violations:")
    if not result.violations:
        lines.append("  - None detected by rule-based scanner.")
    else:
        for i, v in enumerate(result.violations, start=1):
            lines.append(
                f"  {i}. [{v.category}] {v.description} "
                f"(span: '{v.span}')"
            )
    lines.append("")
    lines.append("Safe, Compliant Rewrite:")
    lines.append(result.safe_rewrite)
    lines.append("")
    lines.append("Recommendations:")
    if not result.recommendations:
        lines.append("  - No recommendations returned.")
    else:
        for r in result.recommendations:
            lines.append(f"  - {r}")
    lines.append("")
    lines.append("Narrative Summary:")
    lines.append(result.narrative_summary)
    lines.append("")
    lines.append(f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(result.timestamp))}")
    return "\n".join(lines)


# ============================================================
# SIMPLE EVALUATION UTILS (OPTIONAL)
# ============================================================

def evaluate_agent_on_examples(examples: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    examples: list of dicts with keys:
      - text (str)
      - expected_risk_level (optional)
    Returns DataFrame with model risk vs expected.
    """
    records = []
    for ex in examples:
        res = agent.analyze_text(ex["text"], session_id="evaluation")
        record = {
            "text": ex["text"],
            "risk_score": res.risk_score,
            "risk_level": res.risk_level,
            "expected_risk_level": ex.get("expected_risk_level", None),
        }
        records.append(record)
    df = pd.DataFrame(records)
    return df


# Example eval usage (you can run this in a separate cell):
# examples = [
#     {"text": "Hi team, here's my phone number +1 555-123-4567", "expected_risk_level": "LOW"},
#     {"text": "User credit card: 4111 1111 1111 1111, CVV 123", "expected_risk_level": "HIGH"},
#     {"text": "Public announcement: We will launch our product next week.", "expected_risk_level": "NONE"},
# ]
# eval_df = evaluate_agent_on_examples(examples)
# eval_df


# ============================================================
# GRADIO UI (DEPLOYMENT DEMO)
# ============================================================

def gradio_infer(text: str, session_id: str) -> str:
    if not text.strip():
        return "Please enter some text to audit."
    result = agent.analyze_text(text, session_id=session_id)
    report = format_report(result)

    # Add metrics at bottom for transparency
    metrics = session_memory.get_metrics()
    report += "\n\n=== SYSTEM METRICS (GLOBAL) ===\n"
    report += f"Total requests processed: {metrics['total_requests']}\n"
    report += f"Average risk score: {metrics['avg_risk_score']:.2f}\n"
    return report


# Build Gradio app
with gr.Blocks(title="AI Compliance Audit Agent") as demo:
    gr.Markdown(
        """
        # 🛡️ AI Compliance Audit Agent
        Paste any text, and this agent will:
        - Detect potential privacy / policy violations
        - Assign a risk score (0–100)
        - Generate a safe, compliant rewrite
        - Provide recommendations and a narrative summary
        """
    )

    with gr.Row():
        input_text = gr.Textbox(
            label="Input Text",
            placeholder="Paste email, document excerpt, chat log, etc.",
            lines=10,
        )

    session_id_box = gr.Textbox(
        label="Session ID (optional)",
        value="demo-session",
        info="Use the same ID to keep a short audit history.",
    )

    run_button = gr.Button("Run Compliance Audit", variant="primary")

    output_report = gr.Textbox(
        label="Audit Report",
        lines=20,
    )

    run_button.click(
        fn=gradio_infer,
        inputs=[input_text, session_id_box],
        outputs=[output_report],
    )

# To launch inside Kaggle, uncomment this line:
# demo.launch()


print("✅ AI Compliance Audit Agent code cell executed. You can now run demo.launch() to start the UI.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 7.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


[2025-11-16 10:56:07,690] [WARNING] COHERE_API_KEY not set – using MOCK LLM responses for testing only.


✅ AI Compliance Audit Agent code cell executed. You can now run demo.launch() to start the UI.
